### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="mercari_price_suggestion",
    dataset_year="2018",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/mercari-price-suggestion-challenge",
    download_description="""
train.tsv.7z

kaggle competitions download -c mercari-price-suggestion-challenge -f train.tsv.7z && 7z x train.tsv.7z && rm train.tsv.7z
mkdir -p local-data-warehouse/mercari_price_suggestion && mv train.tsv local-data-warehouse/mercari_price_suggestion
""",
    # References
    academic_reference_bibtex=r"""@misc{Howard2017MercariPriceSuggestionChallenge,
  author = {{Kaggle} and Addison Howard and kaoriiida and Kei Otagaki and Mark McDonald and mueno and Wendy Kan and Zhang and zyaga},
  title  = {Mercari Price Suggestion Challenge},
  year   = {2017},
  howpublished = {\url{https://kaggle.com/competitions/mercari-price-suggestion-challenge}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Howard2017MercariPriceSuggestionChallenge",
    license="Kaggle Competition Rules",
    data_tags=["IID"],
    curation_comments="""
We start from the Kaggle competition dataset.

- We log1p scale the target to match the target metric of the competition by default RMSE.
- The dataset was solved by Kaggle experts by tabular models and a lot of custom string preprocessing. So it is a great use case to test encoding pipelines! The dataset also has mostly string data.
- The task was treated and solved as an IID task on Kaggle.
- We replace "No description yet" with NaN.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="price",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.tsv", sep="\t")
print("Loaded data shape:", df.shape)

df["price"] = np.log1p(df["price"])
df = df.drop(columns=["train_id"])
df["item_description"] = df["item_description"].replace("No description yet", np.nan)

as_cat_dtype = [
    "item_condition_id",
    "shipping",
]
as_string_dtype = [
    "name",
    "category_name", # multi-categorical, not just one category name!
    "brand_name",
    "item_description",
]

for c in as_string_dtype:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_dtype] = df[as_cat_dtype].astype("category")

# Drop duplicates
df = df.drop_duplicates()

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (1482535, 8)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
    duplicate_column_check=False, # skip, no duplicates
)


#### Dataset Overview
Rows: 1,482,486
Columns: 7
Use sampling: True (sample size: 148,249)
Get missing and unique counts per column...


missing/unique per-col:   0%|          | 0/7 [00:00<?, ?it/s]

Get example values per column...


examples per-col:   0%|          | 0/7 [00:00<?, ?it/s]

Get numeric feature statistics...


numeric stats:   0%|          | 0/1 [00:00<?, ?it/s]

Get cat stats...


cat stats:   0%|          | 0/6 [00:00<?, ?it/s]

Get target stats...
Get row duplicates (staged, merged)...
Using top-6 columns for initial filtering: ['item_description', 'name', 'brand_name', 'category_name', 'item_condition_id', 'shipping']


Rows remaining as candidates after top-6 filter: 183 (of 1,482,486)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 94 (0.01% of dataset)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,POPULAR STYLE HEADBAND FREE SH,1,Women/Women's Accessories/Hair Accessories,<NA>,2.772589,1,Designer style Great grip Free fast shipping Package sealed no box
1,bundle for hollywood,3,Women/Other/Other,<NA>,4.700480,0,bundle for hollywood
2,Levi capris sz 12,3,"Women/Pants/Capris, Cropped",Levi's®,2.564949,0,"White Capri pants by Levi size 12. In great condition barely worn maybe only a couple times. Comes from clean, smoke free home. Bundle & save! If you are going to buy more then one item, let me know so I can make a custom listing where you can save on shipping cost."
3,The trash pack - purple series,3,Kids/Other/Other,<NA>,3.295837,0,1 large trash pack with seven characters 22 small trash pack with one character each Check out my other listing for more trash packs.
4,VS PINK Jacket,3,Women/Athletic Apparel/Jackets,PINK,2.995732,0,pink vs pink jacket! zips and buttons. fur on the inside


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,item_condition_id,category,0.0,0.00,5.0,"1, 3, 2, 4, 5"
1,shipping,category,0.0,0.00,2.0,"0, 1"
2,price,float64,0.0,0.00,828.0,"2.3979, 2.5649, 2.7081, 2.8332, 2.3026, 2.1972, 2.7726, 3.0445, 2.0794, 3.2189"
3,brand_name,string,632641.0,42.67,4809.0,"PINK, Nike, Victoria's Secret, LuLaRoe, Apple, FOREVER 21, Nintendo, Lululemon, American Eagle, Michael Kors"
4,item_description,string,82495.0,5.56,1281424.0,"New, Brand new, Great condition, Good condition, NWT, New with tags, Like new, Never worn, Never used, Worn once"
5,category_name,string,6327.0,0.43,1287.0,"Women/Athletic Apparel/Pants, Tights, Leggings, Women/Tops & Blouses/T-Shirts, Beauty/Makeup/Face, Beauty/Makeup/Lips, Electronics/Video Games & Consoles/Games, Electronics/Cell Phones & Accessories/Cases, Covers & Skins, Beauty/Makeup/Eyes, Women/Underwear/Bras, Women/Tops & Blouses/Tank, Cami, Women/Dresses/Above Knee, Mini"
6,name,string,0.0,0.00,1225273.0,"Bundle, Coach purse, Converse, Romper, American Eagle Jeans, BUNDLE, Reserved, Dress, Vans, Lularoe TC leggings"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
price,148249.0,2.977075,0.749001,0.0,7.504392


In [7]:
# Categorical Feature Statistics
cat_stats

value  count  \
column            rank                                                          
brand_name        1                                               <NA>  63577   
                  2                                               PINK   5297   
                  3                                               Nike   5295   
                  4                                  Victoria's Secret   4758   
                  5                                            LuLaRoe   3029   
category_name     1     Women/Athletic Apparel/Pants, Tights, Leggings   6009   
                  2                      Women/Tops & Blouses/T-Shirts   4541   
                  3                                 Beauty/Makeup/Face   3448   
                  4                                 Beauty/Makeup/Lips   3016   
                  5           Electronics/Video Games & Consoles/Games   2635   
item_condition_id 1                                                  1  63754   
                  2                                                  3  43458   
                  3                                                  2  37543   
                  4                                                  4   3244   
                  5                                                  5    250   
item_description  1                                               <NA>   8206   
                  2                                                New    436   
                  3                                          Brand new    283   
                  4                                    Great condition    144   
                  5                                     Good condition    129   
name              1                                             Bundle    216   
                  2                                        Coach purse     46   
                  3                                           Converse     44   
                  4                                             Romper     43   
                  5                               American Eagle Jeans     42   
shipping          1                                                  0  81890   
                  2                                                  1  66359   

                          pct  
column            rank         
brand_name        1     42.89  
                  2      3.57  
                  3      3.57  
                  4      3.21  
                  5      2.04  
category_name     1      4.05  
                  2      3.06  
                  3      2.33  
                  4      2.03  
                  5      1.78  
item_condition_id 1      43.0  
                  2     29.31  
                  3     25.32  
                  4      2.19  
                  5      0.17  
item_description  1      5.54  
                  2      0.29  
                  3      0.19  
                  4       0.1  
                  5      0.09  
name              1      0.15  
                  2      0.03  
                  3      0.03  
                  4      0.03  
                  5      0.03  
shipping          1     55.24  
                  2     44.76

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.06,0.662,-0.148,0.561,0.035,log1p,692149.5,491781.1,lognormal


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=1, n_splits=1, test_size=250000


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to mercari_price_suggestion/019d736e-d881-72ea-b2fb-663546914423


019d736e-d881-72ea-b2fb-663546914423
9164b2d6a461f996a6de173ebaa173d922fc9b3db7a30753bdeab86e89efbec2
